# 01 · From a language model to an application

**Session 01 · Building AI applications for research and data · notebook 1 of 6**

## Objective

This notebook shows what a language model does with a research question when it receives nothing but the question, and what changes when the application gives it instructions and evidence. The purpose is to establish the first two components of an application, the instructions and the context, and to explain why both belong to the application.

## The application after this notebook

Diagram: the question and the context feed the prompt, which goes to the model and produces the answer.

The prompt is the only component that this notebook builds. The question and the context go into the prompt, the prompt goes to the model, and the model produces the answer. Each later notebook adds one component around this core: a document store, a retrieval step, data tools, an agent loop, and a verifier.

The whole session works on one research question, so that every method can be judged on the same case:

> How have internet access and economic development evolved in Rwanda and Kenya? Compare the recent available data, summarize the main patterns, and cite the sources.

## Contents

1. [Section 1](#scrollTo=section-1) defines the prompt and its three parts.
2. [Section 2](#scrollTo=section-2) sends the question to the model with no evidence. The answer is well organized and carefully hedged, but no figure is tied to a source or a year, so none of them can be checked.
3. [Section 3](#scrollTo=section-3) supplies four country briefs as context. The answer becomes specific and cites each figure.
4. [Section 4](#scrollTo=section-4) asks a question that the briefs cannot answer and shows how an instruction makes the model say so.
5. [Section 5](#scrollTo=section-5) asks a question with two meanings under two sets of instructions. With the second set, the model asks which meaning is intended before it answers.
6. [Section 6](#scrollTo=section-6) prints the messages exactly as the model received them.
7. [Section 7](#scrollTo=section-7) measures what the context cost in tokens and explains why a whole document collection cannot be supplied in the same way.

## How to run it

Run the cells in order. Without an API key, the model's outputs are recorded ones, labelled `[recorded output: ...]`. Each cell names its recorded scenario with `w.use(...)` before it calls the model. With a key in `WORKSHOP_API_KEY`, the same cells call the model live, and the answers will differ in wording from the recorded ones.

The next cell installs the session's tools and loads its helper module. It takes about a minute the first time, and you do not need to read it.

In [1]:
# Setup. In Colab this clones the repository and installs the dependencies; locally it uses the folder as is.
import os, sys, subprocess, pathlib

REPO_URL = "https://github.com/worldbank/ai4data.git"   # the program's repository
BRANCH = "main"   # the branch the Colab setup clones
SESSION = "training/session-01-building-ai-applications/notebooks"

if not pathlib.Path("workshop.py").exists():
    subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", BRANCH, REPO_URL, "/content/ai4data"], check=True)
    os.chdir(f"/content/ai4data/{SESSION}")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
sys.path.insert(0, os.getcwd())

# Optional: a key for an OpenAI-compatible API. Without one, the notebook replays recorded outputs and says so.
try:
    from google.colab import userdata  # type: ignore
    for name in ("WORKSHOP_API_KEY", "WORKSHOP_BASE_URL", "WORKSHOP_MODEL"):
        if userdata.get(name):
            os.environ[name] = userdata.get(name)
except Exception:
    pass

import workshop as w
model = w.make_model()
print("Model:", w.mode(model))

Model: recorded outputs (no WORKSHOP_API_KEY set)


## The stack

| Need | Library |
|---|---|
| Documents, splitting, embeddings, vector store, prompts, tools, structured output | [LangChain](https://python.langchain.com/) |
| The agent loop | [LangGraph](https://langchain-ai.github.io/langgraph/), through `langchain.agents.create_agent` |
| Tool server and client over the Model Context Protocol | [FastMCP](https://gofastmcp.com/) |
| Number verification | [`pcn-core`](https://github.com/worldbank/pcn), the Python core of the Proof-Carrying Numbers protocol |
| Output schemas | [Pydantic](https://docs.pydantic.dev/) |

`workshop.py` holds what these do not: the worked-example data, a chat model that replays recorded outputs when no API key is configured, and a few printing and evaluation helpers.

## 1. The three parts of a prompt

Every call to the model sends one prompt, and the prompt is assembled from three parts.

- **Instructions** are the system message. They tell the model how to behave: what its role is, how it should cite evidence, and what it should do when the evidence does not contain the answer. The application's builders write them.
- **Context** is the evidence that the application supplies with the question, such as the documents or the data records that the model should answer from. The application selects it for each question.
- **The question** is the text that the user types.

The three parts are kept in separate slots because a different person changes each of them for a different reason. The builders revise the instructions when the behaviour needs to change. The application fills the context for every question. The user supplies the question. LangChain's `ChatPromptTemplate` holds the three slots, and the expression `prompt | model` is the whole application at this stage.

Animation: instructions set by the developer, context supplied by the application and a question asked by the user are combined into one prompt, which goes to the model and produces an answer that cites its source.

*The animation loops. Its text comes from this notebook: the instructions are from the cell below, the context and the answer are from section 3, and the question is the worked example. Section 6 shows the same three parts as the model received them.*

In [2]:
from langchain_core.prompts import ChatPromptTemplate

INSTRUCTIONS_BASIC = "You are a research assistant. Answer the question."

INSTRUCTIONS_GROUNDED = """You are a research assistant for development data.
Answer only from the supplied context. Cite the document id in square brackets after each figure.
If the context does not contain what is needed, say so and give the closest available information."""

INSTRUCTIONS_CLARIFY = INSTRUCTIONS_GROUNDED + """
If the question is ambiguous, say which readings are possible and ask which one is meant before answering."""

prompt = ChatPromptTemplate.from_messages([
    ("system", "{instructions}"),
    ("human", "CONTEXT:\n{context}\n\nQUESTION: {question}"),
])
chain = prompt | model

w.show(w.MAIN_QUESTION)

How have internet access and economic development evolved in Rwanda and Kenya? Compare the recent available data, summarize the main patterns, and cite the sources.

## 2. The model without context

This section establishes the baseline. The cell sends the question with the basic instructions and no evidence, so the model can only answer from what it absorbed during training.

In [3]:
w.use("nb01_zero_context")
answer = chain.invoke({"instructions": INSTRUCTIONS_BASIC, "context": "(none)", "question": w.MAIN_QUESTION})
w.show(answer)

`[recorded output: nb01_zero_context]`

# Internet Access and Economic Development in Rwanda and Kenya

I can't access live databases, so the figures below come from my training data. They are approximate and may be outdated, and definitions differ across sources (for example, subscriptions versus unique users). Check the latest releases before citing numbers in formal work.

## Snapshot comparison (approximate, ~2022–2024)

| Indicator | Kenya | Rwanda |
|---|---|---|
| Population | ~55 million | ~14 million |
| GDP (current US$) | ~$110–115 billion | ~$13–14 billion |
| GDP per capita | ~$2,000–2,200 | ~$950–1,050 |
| Real GDP growth | ~5% (2023), ~4–5% (2024) | ~8% (2023), ~9–10% (2024) |
| Internet users (share of population) | roughly 40% | roughly 30–40% (estimates vary widely) |
| Mobile broadband | Large base of mobile data subscriptions, widely used | Expanding; 4G population coverage is high |
| Mobile money | Very high adoption (M-Pesa is a global reference case) | High adoption, growing via national platforms |
| Human Development Index | ~0.58 (mid-tier) | ~0.53 (low-medium tier, improving) |
| Poverty | ~36% national poverty line (2020 survey) | Fell from ~77% (2001) to ~38% (2016/17); further decline reported in later surveys |

## Main patterns

**1. Both countries have expanded access quickly, mostly through mobile.** Mobile phones, not fixed broadband, drive connectivity in both. Fixed broadband and household internet access remain low, especially outside cities.

**2. Kenya's model is market-led and mobile-money-driven.** Competition among mobile operators, a dense private sector, and M-Pesa have made Kenya a reference case for mobile-enabled finance and digital services. Its growth has been broad-based, though poverty and inequality remain high.

**3. Rwanda's model is state-led and infrastructure-focused.** The government built a national fiber backbone, pursued universal coverage targets, and promoted digital services through institutions such as Smart Rwanda and the Kigali Innovation City initiative. Its growth has been faster, but from a lower income base.

**4. Affordability is a binding constraint in both countries.** Data prices relative to income remain high for many households, so access does not automatically translate into use. Usage gaps are largest among women, rural residents, and low-income groups.

**5. Connectivity and growth move together, though causality is hard to isolate.** Cross-country studies consistently link broadband and mobile adoption to productivity and GDP growth, and both countries' digital strategies assume this relationship. Disentangling the effect of internet access from other reforms in these two countries requires careful econometric work.

## Suggested sources

- **ITU**, *Facts and Figures* (2024) and *Measuring Digital Development: Facts and Figures* (ITU DataHub): internet use and affordability indicators.
- **World Bank**, World Development Indicators (databank.worldbank.org): GDP, GDP per capita, poverty, and growth.
- **IMF**, *World Economic Outlook* (October 2024 edition): growth forecasts and country data.
- **Kenya National Bureau of Statistics (KNBS)**, *Economic Survey 2024* and *Kenya Poverty Report*.
- **Communications Authority of Kenya**, *Quarterly Sector Statistics Reports*: mobile and internet subscriptions. Note that these count subscriptions, not individuals.
- **Rwanda Utilities Regulatory Authority (RURA)**, annual reports and ICT sector statistics.
- **National Institute of Statistics of Rwanda (NISR)**, *Integrated Household Living Conditions Survey (EICV)* and ICT household surveys: poverty and household internet access.
- **Smart Rwanda Master Plan 2050** and related Rwandan ICT strategy documents.
- **GSMA**, *The Mobile Economy Sub-Saharan Africa* (annual): mobile connectivity and economic contribution.
- **Alliance for Affordable Internet (A4AI)**, *Affordability Report*: data-price comparisons.
- **DataReportal** (*Digital 2024/2025* country reports): internet user estimates.
- **UNDP**, *Human Development Report*: HDI values.

If you tell me which years or indicators matter most for your purpose, I can narrow the comparison and suggest the most appropriate source for each figure.

The answer cannot be judged without reference figures. They come from the World Bank's World Development Indicators, which the model did not see. The next two cells put the model's claims beside those figures. The first cell prints the official values for Kenya and Rwanda: the share of people who use the internet, and GDP per capita. The second cell takes six claims from the answer and sets each one beside the matching official figure.

In [4]:
rows = []
for ind in ["IT.NET.USER.ZS", "NY.GDP.PCAP.CD"]:
    fmt = "{:,.0f}" if ind == "NY.GDP.PCAP.CD" else "{:.1f}"
    for country in ["Kenya", "Rwanda"]:
        r = w.indicator_lookup(ind, country, start=2023)
        by_year = {o["year"]: fmt.format(o["value"]) for o in r["observations"]}
        rows.append((r["name"], country, by_year[2023], by_year[2024]))
w.show_table(["Indicator", "Country", "2023", "2024"], rows, source="Source: World Bank, World Development Indicators.")

Indicator,Country,2023,2024
Individuals using the Internet (% of population),Kenya,32.1,35.0
Individuals using the Internet (% of population),Rwanda,28.8,31.7
GDP per capita (current US$),Kenya,"1,943","2,133"
GDP per capita (current US$),Rwanda,"1,061","1,060"


In [5]:
def value(ind, country):
    o = w.latest(w.indicator_lookup(ind, country, start=2023))
    return o["value"], o["year"]


def growth(country, year):
    return next(o["value"] for o in w.indicator_lookup("NY.GDP.MKTP.KD.ZG", country, start=year)["observations"] if o["year"] == year)


rows = [  # (claim, what the model said, what the data says, verdict)
    ("Kenya · internet use", "“roughly 40%” of the population", "{:.1f}% of people online in {}".format(*value("IT.NET.USER.ZS", "Kenya")), "above the data"),
    ("Rwanda · internet use", "“roughly 30–40% (estimates vary widely)”", "{:.1f}% of people online in {}".format(*value("IT.NET.USER.ZS", "Rwanda")), "close"),
    ("Kenya · GDP per capita", "“~$2,000–2,200”", "{:,.0f} US dollars in {}".format(*value("NY.GDP.PCAP.CD", "Kenya")), "close"),
    ("Rwanda · GDP per capita", "“~$950–1,050”", "{:,.0f} US dollars in {}".format(*value("NY.GDP.PCAP.CD", "Rwanda")), "close"),
    ("Kenya · GDP growth", "“~5% (2023), ~4–5% (2024)”", f"{growth('Kenya', 2023):.1f} percent in 2023, {growth('Kenya', 2024):.1f} percent in 2024", "close"),
    ("Rwanda · GDP growth", "“~8% (2023), ~9–10% (2024)”", f"{growth('Rwanda', 2023):.1f} percent in 2023, {growth('Rwanda', 2024):.1f} percent in 2024", "2024 above the data"),
]
w.show_claim_check(rows)

Claim,What the model said,What the data says,Verdict,Source named
Kenya · internet use,“roughly 40%” of the population,35.0% of people online in 2024,above the data,none
Rwanda · internet use,“roughly 30–40% (estimates vary widely)”,31.7% of people online in 2024,close,none
Kenya · GDP per capita,"“~$2,000–2,200”","2,133 US dollars in 2024",close,none
Rwanda · GDP per capita,"“~$950–1,050”","1,060 US dollars in 2024",close,none
Kenya · GDP growth,"“~5% (2023), ~4–5% (2024)”","5.7 percent in 2023, 4.7 percent in 2024",close,none
Rwanda · GDP growth,"“~8% (2023), ~9–10% (2024)”","8.6 percent in 2023, 7.2 percent in 2024",2024 above the data,none


The answer is careful. It says that its figures come from training data, warns that they are approximate, distinguishes subscriptions from users, and ends with a reading list. Four of the six claims in the table land close to the official figures. Two do not: the share of Kenyans online is given as roughly 40 percent against 35.0 percent in 2024, and Rwanda's growth in 2024 is given as 9 to 10 percent against 7.2 percent. The reader has no way to tell these two from the other four. Every figure is a rounded range, no figure is tied to a year, and the sources at the end are institutions to consult, so no figure can be traced to a record. The hedging is correct behaviour for a model that has no evidence in front of it, and it leaves a reader who needs a citable number with the whole research task still to do.

Different instructions would change the tone and the length of this answer, but the figures would still come from training, because the model has no other source.

## 3. The same question with context

This section supplies evidence. The four country briefs from the session's document collection go into the context slot, and the instructions change to the grounded set, which requires a citation after each figure and allows the model to say when something is missing. The cell prints the size of the context before it calls the model.

In [6]:
docs = {d.metadata["id"]: d for d in w.load_documents()}
context = "\n\n".join(f"[{i}]\n{docs[i].page_content}" for i in ["kenya-digital", "rwanda-digital", "kenya-economy", "rwanda-economy"])
print(f"{len(context):,} characters of context, roughly {len(context)//4:,} tokens\n")

w.use("nb01_with_context")
answer = chain.invoke({"instructions": INSTRUCTIONS_GROUNDED, "context": context, "question": w.MAIN_QUESTION})
w.show(answer)

4,098 characters of context, roughly 1,024 tokens



`[recorded output: nb01_with_context]`

Internet use. In Kenya, the share of individuals using the internet rose from 7.2 percent of the population in 2010 to 35.0 percent in 2024 [kenya-digital]. In Rwanda it rose from 8.0 percent in 2010 to 31.7 percent in 2024 [rwanda-digital]. Kenya's fastest growth came between 2010 and 2015; Rwanda's came between 2015 and 2020. Both countries ended the period with about a third of the population online.

Economic development. Kenya's GDP per capita in current US dollars rose from 1,092 in 2010 to 2,133 in 2024, with a dip to 1,943 in 2023 that coincided with a currency depreciation [kenya-economy]. Rwanda's rose from 587 in 2010 to 1,060 in 2024, passing 1,000 in 2022 [rwanda-economy]. Rwanda grew faster in most years, above 7 percent in 10 of the 15 and contracting only in 2020, while Kenya grew in every year except 2020.

Patterns. Kenya has the higher level on both indicators; Rwanda has the faster growth on both. Internet use in the two countries converged, from a gap of about 4 percentage points in 2015 to about 3 points in 2024, while the gap in GDP per capita remained close to two to one.

Sources: World Development Indicators, indicators IT.NET.USER.ZS and NY.GDP.PCAP.CD, as reproduced in the supplied briefs.

Diagram: the same question without context gets an answer with no source, no date and a blend of two measures; with the four country briefs as context it gets a dated answer that cites its source.

*The two answers, shortened. They are the recorded outputs of sections 2 and 3.*

The figures now come from the briefs, each figure carries a citation, and the comparison between the two countries is specific. The model's part in this improvement is small. The application chose the four briefs and placed them in the prompt, and the model wrote from that text. This division of labour holds for the rest of the session: the application decides what evidence the model sees, and the quality of the answer follows from that choice.

## 4. A question the evidence cannot answer

An application also has to behave well when the evidence does not contain the answer. The cell asks for Kenya's internet use in 2025, and the brief only goes up to 2024. The grounded instructions tell the model to say that the information is missing and to give the closest available figure, so the answer should name 2024 and say that 2025 is absent from the evidence.

In [7]:
w.use("nb01_missing_info")
answer = chain.invoke({"instructions": INSTRUCTIONS_GROUNDED, "context": docs["kenya-digital"].page_content,
                       "question": "What share of Kenya's population used the internet in 2025?"})
w.show(answer)

`[recorded output: nb01_missing_info]`

The supplied material contains no observation for 2025. The latest available year for the share of Kenya's population using the internet is 2024, at 35.0 percent [kenya-digital]. If a 2025 value is needed, it will have to come from a later release of the World Development Indicators or from the ITU.

## 5. An ambiguous question under two sets of instructions

"How connected is Kenya?" can mean internet use, mobile subscriptions, broadband coverage, or something else. The cell asks it twice: once with the basic instructions, and once with instructions that tell the model to list the possible readings and ask which one is meant.

In [8]:
print("Without an instruction to clarify:\n")
w.use("nb01_ambiguous_no_instruction")
w.show(chain.invoke({"instructions": INSTRUCTIONS_BASIC, "context": docs["kenya-digital"].page_content, "question": "How connected is Kenya?"}))
print("\n" + "=" * 100 + "\nWith the instruction:\n")
w.use("nb01_ambiguous_with_instruction")
w.show(chain.invoke({"instructions": INSTRUCTIONS_CLARIFY, "context": docs["kenya-digital"].page_content, "question": "How connected is Kenya?"}))

Without an instruction to clarify:



`[recorded output: nb01_ambiguous_no_instruction]`

Kenya is highly connected. Mobile cellular subscriptions reached 126.5 per 100 people in 2024, which is more than one subscription per person, and 35.0 percent of the population used the internet in the same year. Kenya is also the home of M-Pesa, so mobile money use is very high. Overall, Kenya is one of the most connected countries in East Africa.


With the instruction:



`[recorded output: nb01_ambiguous_with_instruction]`

"Connected" can mean several things, and the supplied material measures two of them. Which do you mean?

- Share of individuals using the internet: 35.0 percent of the population in 2024 [kenya-digital].
- Mobile cellular subscriptions: 126.5 per 100 people in 2024, a count of subscriptions that can exceed the number of people [kenya-digital].

The two measures answer different questions. Subscriptions passed one per person around 2020, while internet use stood at about a third of the population in 2024. If you are asking about something else, for example broadband speed or household access, the supplied material does not cover it.

With the basic instructions, the model chooses one reading and answers it with confidence. With the clarifying instructions, the model names the readings and asks before it answers. For a research application, the second behaviour is the one to specify. A confident answer to the wrong reading looks the same on the page as a confident answer to the right one, so the reader cannot detect the substitution. Asking costs one exchange and removes that risk.

## 6. The prompt as sent to the model

The cell prints the messages that the prompt produced for the last call, with each part in its colour from section 1. The instructions arrive as the system message. The context and the question arrive together in the human message.

In [9]:
messages = prompt.format_messages(instructions=INSTRUCTIONS_CLARIFY, context=docs["kenya-digital"].page_content, question="How connected is Kenya?")
w.show_prompt(messages)

## 7. The cost of context

Section 3 reported the size of its context: four short briefs took about a thousand tokens. The obvious extension is to supply every document an organization holds in the same way, so that the model always has everything in front of it. That approach fails for three reasons.

- **The collection does not fit.** A model reads a limited amount of text in one call, called its context window. A collection of thousands of documents is far larger than any window.
- **The cost grows with every page.** The price of a call grows with the number of tokens, and the whole collection would be sent again with every question.
- **The relevant text gets lost.** The passages that answer one question are a small part of the collection, and the rest surrounds them as noise.

The application therefore has to select, for each question, the small part of the collection that the model should read. Notebooks 02 and 03 build that selection: the first prepares the collection, and the second searches it.

Diagram: the four briefs of this notebook, about 1,024 tokens, fit inside the model's context window, while a real collection of thousands of documents cannot, so the application must choose what the model reads.

*The token count is the one printed in section 3.*

## Key points

- A prompt has three parts: the instructions, the context, and the question. They are kept in separate slots because a different person changes each one.
- Without context, the model answers from its training. The answer was well organized and hedged, four of its six figures were close to the official ones, two were off, and no figure was tied to a source or a year, so a reader could not tell which figures to trust.
- With the four briefs as context, the answer became specific and cited each figure. The application produced that improvement by choosing the evidence, and the model wrote from what it was given.
- Instructions shape the model's behaviour. They can require citations, make the model say when information is missing, and make it ask when a question has more than one meaning. They cannot supply data that the model lacks.
- A whole collection cannot be supplied as context. The context window has a limit, the cost grows with every token, and the relevant passages get lost among the rest. The next notebooks build the selection step.

## Exercise

1. Read the answers of sections 2 and 3 side by side. For each number, note whether it can be checked and where. Then decide which answer could be sent to a colleague, and give the reason.
2. In the cell of section 1, add a sentence to `INSTRUCTIONS_GROUNDED`, for example "Answer in no more than three sentences." Run that cell again, then run the cell of section 6. The Instructions block now shows the added sentence exactly as the model would receive it. The recorded answers do not react to new instructions. With a live key in `WORKSHOP_API_KEY`, rerun section 3 to see how the answer changes.

## Where this connects to the program

- [Practical Guide to Serving Official Statistics to AI Agents](https://worldbank.github.io/ai4data/cookbook/serving-statistics-to-agents/): the design of tools over a statistical catalog, in full.
- [Practical Guide to Evaluation Suites for Statistical AI](https://worldbank.github.io/ai4data/cookbook/evaluation-suites/): how to build and run the tests.
- [Proof-Carrying Numbers](https://worldbank.github.io/ai4data/docs/pcn/): the claim-tag protocol the verifier in this session follows. See also the [paper](https://arxiv.org/abs/2509.06902) and the [`pcn` repository](https://github.com/worldbank/pcn).
- [Data360 MCP server](https://github.com/worldbank/data360-mcp): the tools of notebook 04, exposed to any MCP client over the World Bank's data platform.
- [End-to-end reference implementation](https://worldbank.github.io/ai4data/docs/reference-implementation/): documentation, validation, publication, agent access, and verification in one run.

**Next:** [02 · Preparing knowledge for an application](https://colab.research.google.com/github/worldbank/ai4data/blob/main/training/session-01-building-ai-applications/notebooks/02-preparing-knowledge.ipynb)